In [1]:
import pandas as pd
import ast
import numpy as np

In [2]:
INPUT_FILE = "../data/raw/movies_metadata.csv"
OUTPUT_FILE = "../data/processed/cleaned_movies.csv"

In [3]:
print("Loading dataset...")

movies = pd.read_csv(
    INPUT_FILE,
    low_memory=False
)

print("Dataset loaded.")
print("Original shape:", movies.shape)

Loading dataset...
Dataset loaded.
Original shape: (45466, 24)


In [4]:
movies.head()

,adult,belongs_to_collection,budget,genres,homepage,id,imdb_id,original_language,original_title,overview,...,release_date,revenue,runtime,spoken_languages,status,tagline,title,video,vote_average,vote_count
0,False,"{'id': 10194, 'name': 'Toy Story Collection', ...",30000000,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...",http://toystory.disney.com/toy-story,862,tt0114709,en,Toy Story,"Led by Woody, Andy's toys live happily in his ...",...,1995-10-30,373554033.0,81.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,NaN,Toy Story,False,7.7,5415.0
1,False,NaN,65000000,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...",NaN,8844,tt0113497,en,Jumanji,When siblings Judy and Peter discover an encha...,...,1995-12-15,262797249.0,104.0,"[{'iso_639_1': 'en', 'name': 'English'}, {'iso...",Released,Roll the dice and unleash the excitement!,Jumanji,False,6.9,2413.0
2,False,"{'id': 119050, 'name': 'Grumpy Old Men Collect...",0,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...",NaN,15602,tt0113228,en,Grumpier Old Men,A family wedding reignites the ancient feud be...,...,1995-12-22,0.0,101.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Still Yelling. Still Fighting. Still Ready for...,Grumpier Old Men,False,6.5,92.0
3,False,NaN,16000000,"[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...",NaN,31357,tt0114885,en,Waiting to Exhale,"Cheated on, mistreated and stepped on, the wom...",...,1995-12-22,81452156.0,127.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Friends are the people who let you be yourself...,Waiting to Exhale,False,6.1,34.0
4,False,"{'id': 96871, 'name': 'Father of the Bride Col...",0,"[{'id': 35, 'name': 'Comedy'}]",NaN,11862,tt0113041,en,Father of the Bride Part II,Just when George Banks has recovered from his ...,...,1995-02-10,76578911.0,106.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Just When His World Is Back To Normal... He's ...,Father of the Bride Part II,False,5.7,173.0


In [5]:
print("Columns:")
print(movies.columns.tolist())

Columns:
['adult', 'belongs_to_collection', 'budget', 'genres', 'homepage', 'id', 'imdb_id', 'original_language', 'original_title', 'overview', 'popularity', 'poster_path', 'production_companies', 'production_countries', 'release_date', 'revenue', 'runtime', 'spoken_languages', 'status', 'tagline', 'title', 'video', 'vote_average', 'vote_count']


In [6]:
numeric_columns = [
    "budget",
    "revenue",
    "popularity",
    "runtime",
    "vote_average",
    "vote_count"
]

for column in numeric_columns:
    movies[column] = pd.to_numeric(
        movies[column],
        errors="coerce"
    )

print("Numeric columns converted.")

Numeric columns converted.


In [7]:
movies["release_date"] = pd.to_datetime(
    movies["release_date"],
    errors="coerce"
)

movies["release_year"] = movies["release_date"].dt.year
movies["release_month"] = movies["release_date"].dt.month

print("Release date cleaned.")

Release date cleaned.


In [8]:
def extract_names(value):
    if pd.isna(value):
        return []

    try:
        data = ast.literal_eval(value)

        if isinstance(data, list):
            return [
                item["name"]
                for item in data
                if isinstance(item, dict) and "name" in item
            ]

    except (ValueError, SyntaxError, TypeError):
        return []

    return []

In [9]:
movies["genres"] = movies["genres"].apply(extract_names)

print("Genres cleaned.")

Genres cleaned.


In [10]:
movies["production_countries"] = movies[
    "production_countries"
].apply(extract_names)

print("Production countries cleaned.")

Production countries cleaned.


In [11]:
movies["spoken_languages"] = movies[
    "spoken_languages"
].apply(extract_names)

print("Spoken languages cleaned.")

Spoken languages cleaned.


In [12]:
movies[
    [
        "title",
        "genres",
        "production_countries",
        "spoken_languages"
    ]
].head()

,title,genres,production_countries,spoken_languages
0,Toy Story,"[Animation, Comedy, Family]",[United States of America],[English]
1,Jumanji,"[Adventure, Fantasy, Family]",[United States of America],"[English, Français]"
2,Grumpier Old Men,"[Romance, Comedy]",[United States of America],[English]
3,Waiting to Exhale,"[Comedy, Drama, Romance]",[United States of America],[English]
4,Father of the Bride Part II,[Comedy],[United States of America],[English]


In [13]:
movies = movies.drop_duplicates(
    subset=["id"]
)

print("Duplicates removed.")
print("Current shape:", movies.shape)

Duplicates removed.
Current shape: (45436, 26)


In [14]:
movies = movies[
    movies["revenue"].notna()
    & (movies["revenue"] > 0)
]

print("Invalid revenue rows removed.")
print("Current shape:", movies.shape)

Invalid revenue rows removed.
Current shape: (7398, 26)


In [15]:
movies = movies[
    movies["budget"].notna()
    & (movies["budget"] > 0)
]

print("Invalid budget rows removed.")
print("Current shape:", movies.shape)

Invalid budget rows removed.
Current shape: (5375, 26)


In [16]:
movies = movies[
    movies["runtime"].notna()
    & (movies["runtime"] > 0)
]

print("Invalid runtime rows removed.")
print("Current shape:", movies.shape)

Invalid runtime rows removed.
Current shape: (5363, 26)


In [17]:
movies = movies[
    movies["release_date"].notna()
]

print("Movies without release dates removed.")
print("Current shape:", movies.shape)

Movies without release dates removed.
Current shape: (5363, 26)


In [18]:
print("Before removing invalid rows:", movies.shape)
print("After removing invalid rows:", movies.shape)

Before removing invalid rows: (5363, 26)
After removing invalid rows: (5363, 26)


In [19]:
print("Current dataset shape:", movies.shape)

Current dataset shape: (5363, 26)


In [20]:
movies["popularity"] = movies["popularity"].fillna(0)

movies["vote_average"] = movies["vote_average"].fillna(0)

movies["vote_count"] = movies["vote_count"].fillna(0)

movies["original_language"] = movies[
    "original_language"
].fillna("unknown")

print("Missing values handled.")

Missing values handled.


In [21]:
columns = [
    "id",
    "title",
    "budget",
    "revenue",
    "runtime",
    "popularity",
    "vote_average",
    "vote_count",
    "original_language",
    "genres",
    "production_countries",
    "spoken_languages",
    "release_date",
    "release_year",
    "release_month"
]

movies = movies[columns]

print("Required columns selected.")

Required columns selected.


In [22]:
movies.head()

,id,title,budget,revenue,runtime,popularity,vote_average,vote_count,original_language,genres,production_countries,spoken_languages,release_date,release_year,release_month
0,862,Toy Story,30000000.0,373554033.0,81.0,21.946943,7.7,5415.0,en,"[Animation, Comedy, Family]",[United States of America],[English],1995-10-30,1995.0,10.0
1,8844,Jumanji,65000000.0,262797249.0,104.0,17.015539,6.9,2413.0,en,"[Adventure, Fantasy, Family]",[United States of America],"[English, Français]",1995-12-15,1995.0,12.0
3,31357,Waiting to Exhale,16000000.0,81452156.0,127.0,3.859495,6.1,34.0,en,"[Comedy, Drama, Romance]",[United States of America],[English],1995-12-22,1995.0,12.0
5,949,Heat,60000000.0,187436818.0,170.0,17.924927,7.7,1886.0,en,"[Action, Crime, Drama, Thriller]",[United States of America],"[English, Español]",1995-12-15,1995.0,12.0
8,9091,Sudden Death,35000000.0,64350171.0,106.0,5.231580,5.5,174.0,en,"[Action, Adventure, Thriller]",[United States of America],[English],1995-12-22,1995.0,12.0


In [23]:
print("Final dataset shape:", movies.shape)

print("\nFinal columns:")
print(movies.columns.tolist())

Final dataset shape: (5363, 15)

Final columns:
['id', 'title', 'budget', 'revenue', 'runtime', 'popularity', 'vote_average', 'vote_count', 'original_language', 'genres', 'production_countries', 'spoken_languages', 'release_date', 'release_year', 'release_month']


In [24]:
movies.isnull().sum()

id                      0
title                   0
budget                  0
revenue                 0
runtime                 0
popularity              0
vote_average            0
vote_count              0
original_language       0
genres                  0
production_countries    0
spoken_languages        0
release_date            0
release_year            0
release_month           0
dtype: int64

In [25]:
movies.to_csv(
    OUTPUT_FILE,
    index=False
)

print("Cleaned dataset saved to:")
print(OUTPUT_FILE)

Cleaned dataset saved to:
../data/processed/cleaned_movies.csv
